# [Experiment] Parameter-Matched KAN-TabNet | Cosine Annealing LR | Higgs Boson

### Overview
This notebook evaluates the parameter-matched KAN-TabNet architecture under the continuous optimization environment of a Cosine Annealing learning rate schedule.

### Methodological Context: The Final Synthesis
This experiment represents the synthesis of our structural modifications (KAN layers replacing standard linear transformations) and our shifted optimization thermodynamics (CosineLR). By comparing these results directly against the Vanilla CosineLR baseline, we maintain strict experimental controls while evaluating the architecture in a modern learning environment that deliberately departs from the original paper's StepLR approach.

### The Hypothesis
We hypothesize that the learnable B-splines inherent to KAN layers may interact uniquely with the smooth, continuous decay of the Cosine schedule. This notebook investigates whether this gradual "cooling" of the learning rate allows the KAN splines to settle into more optimal, stable configurations compared to the sudden, discrete adjustments forced by the traditional StepLR schedule.

### Setup

In [1]:
%%capture
# install TabNet fork which allows switching between vanilla TabNet and KAN-TabNet
!pip install git+https://github.com/chuo-v/tabnet.git@v1.0.1-kan

In [2]:
import os
import json
import numpy as np
import pandas as pd
import random
import torch

seed = 11
random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

### Dataset Fetching

In [4]:
from sklearn.model_selection import train_test_split
from pytorch_tabnet.tab_model import TabNetClassifier
import gc

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00280/HIGGS.csv.gz'
df = pd.read_csv(url, header=None)

# Column 0 is the class label (1 for signal, 0 for background). Columns 1-28 are features.
X = df.iloc[:, 1:].values
y = df.iloc[:, 0].values.astype(int)

# Free up the massive raw DataFrame from memory
del df
gc.collect()

# divide dataset into 80% train, 20% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=seed, stratify=y
)

# divide temp into 10% validation and 10% test
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)

print(f"\nFinal Train shape: {X_train.shape}")
print(f"Final Valid shape: {X_valid.shape}")
print(f"Final Test shape:  {X_test.shape}\n")


Final Train shape: (8800000, 28)
Final Valid shape: (1100000, 28)
Final Test shape:  (1100000, 28)



### Model Training

In [5]:
MAX_EPOCHS = 500

In [6]:
# Hyperparameters from original paper (TabNet-S model).
# Note: momentum is 1 - 0.6 (paper m_B) to align with PyTorch's BatchNorm API.
paper_config = {
    'n_steps': 5,
    'gamma': 1.5,
    'n_independent': 2,
    'n_shared': 2,
    'lambda_sparse': 1e-6,
    'momentum': 0.4,
    'optimizer_fn': torch.optim.Adam,
    'mask_type': 'sparsemax'
}

clf_kan = TabNetClassifier(
    n_d=9, # adjusted so parameter count is matched with vanilla TabNet
    n_a=9, # adjusted so parameter count is matched with vanilla TabNet
    **paper_config,
    clip_value=2.,
    optimizer_params=dict(lr=0.0025), # reduced as 0.02 lr is too aggressive
    scheduler_fn=torch.optim.lr_scheduler.CosineAnnealingLR,
    scheduler_params=dict(T_max=MAX_EPOCHS, eta_min=1e-5),
    use_kan=True,
    kan_grid_size=3,
    kan_spline_order=3,
    seed=seed,
    verbose=25
)

In [7]:
# Hyperparameters from original paper (TabNet-S model).
paper_fit_config = {
    'batch_size': 16384,
    'virtual_batch_size': 512,
}

clf_kan.fit(
    X_train=X_train, y_train=y_train,
    eval_set=[(X_valid, y_valid)],
    eval_name=['valid'],
    eval_metric=['accuracy'],
    **paper_fit_config,
    max_epochs=MAX_EPOCHS,
    patience=75,
    num_workers=os.cpu_count(),
    drop_last=False,
    compute_importance=False
)

epoch 0  | loss: 0.6222  | valid_accuracy: 0.6846  |  0:01:48s
epoch 25 | loss: 0.50777 | valid_accuracy: 0.74577 |  0:46:48s
epoch 50 | loss: 0.48731 | valid_accuracy: 0.75958 |  1:31:52s
epoch 75 | loss: 0.4781  | valid_accuracy: 0.76442 |  2:16:57s
epoch 100| loss: 0.47428 | valid_accuracy: 0.76688 |  3:02:05s
epoch 125| loss: 0.47253 | valid_accuracy: 0.76739 |  3:47:07s
epoch 150| loss: 0.47099 | valid_accuracy: 0.76779 |  4:32:10s
epoch 175| loss: 0.47008 | valid_accuracy: 0.76874 |  5:17:16s
epoch 200| loss: 0.46943 | valid_accuracy: 0.76961 |  6:02:19s
epoch 225| loss: 0.46893 | valid_accuracy: 0.7694  |  6:47:22s
epoch 250| loss: 0.46855 | valid_accuracy: 0.76945 |  7:32:29s
epoch 275| loss: 0.46821 | valid_accuracy: 0.76859 |  8:17:37s
epoch 300| loss: 0.46784 | valid_accuracy: 0.7702  |  9:02:46s
epoch 325| loss: 0.46761 | valid_accuracy: 0.77007 |  9:47:56s
epoch 350| loss: 0.46741 | valid_accuracy: 0.77029 |  10:33:07s
epoch 375| loss: 0.46719 | valid_accuracy: 0.77044 |  

In [8]:
# sum up all learnable weights in the PyTorch network
param_count = sum(p.numel() for p in clf_kan.network.parameters() if p.requires_grad)

print(f"Total Learnable Parameters: {param_count:,}")

Total Learnable Parameters: 78,854


### Test Evaluation

In [9]:
from sklearn.metrics import accuracy_score

# evaluate on the hold-out test set
preds = clf_kan.predict(X_test)
test_acc = accuracy_score(y_test, preds)

print(f"Test Accuracy: {test_acc:.5f}")

Test Accuracy: 0.77040


### Data Export

In [10]:
BASE_DIR = './kan-tabnet-experiments'
TABLES_DIR = f'{BASE_DIR}/results/higgs_boson/tables'
MODELS_DIR = f'{BASE_DIR}/results/higgs_boson/models'

os.makedirs(TABLES_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# package all metrics into a single JSON payload
experiment_results = {
    "model_type": "KAN-TabNet",
    "dataset": "Higgs Boson",
    "parameters": param_count,
    "scheduler": "CosineAnnealingLR",
    "final_test_accuracy": test_acc,
    "best_epoch": clf_kan.best_epoch,
    "training_history": clf_kan.history.history
}

# save JSON metrics
JSON_FILEPATH = f'{TABLES_DIR}/04_kan_param_matched_cosine_lr_metrics.json'
with open(JSON_FILEPATH, 'w') as f:
    json.dump(experiment_results, f, indent=4)
print(f"Metrics successfully saved to {JSON_FILEPATH}")

# save the model weights
_ = clf_kan.save_model(f'{MODELS_DIR}/04_kan_param_matched_cosine_lr_{param_count // 1000}k')

Metrics successfully saved to ./kan-tabnet-experiments/results/higgs_boson/tables/04_kan_param_matched_cosine_lr_metrics.json
Successfully saved model at ./kan-tabnet-experiments/results/higgs_boson/models/04_kan_param_matched_cosine_lr_78k.zip
